# NB09 v4 — Lean Regulatory Agent (Q5 · Q9 · Q10)

**Scientific AI Builder | targeted ablation | 2026-10-21**

The frozen **NB09 v3 remains unchanged**. This notebook uses the existing *five-law snapshot corpus* and the same Q1–Q10 benchmark without including human scoring keys or prior answers in model context.

**Design:** deterministic exact-law retrieval (0 LLM calls) → one grounded synthesis call → deterministic citation/coverage checks → **at most one** corrective call. It avoids the v3 planner, semantic validator, issue tracker, and LLM auditor. This design should reduce model calls, but **real latency/cost improvements must be measured**.

**Experiment caveat:** The three routes were selected after Q5/Q9/Q10 error analysis. These are **development-set** questions. Do not present improved scores on them as independent generalization. Check unseen food-regulation cases later.

**Sources:** `data/chunks_legal_aware.json`, `data/benchmark_questions_v1.json`. Snapshot evidence is **not** proof of legally effective current law.

In [5]:
# Resolve paths from the repository root, including nested Jupyter kernel CWDs.
import os as _repo_os
from pathlib import Path as _RepoPath
_repo_start = _RepoPath(_repo_os.environ.get("REGULATORY_AGENT_ROOT", _RepoPath.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_repo_start, *_repo_start.parents)
                  if (p / "data/benchmark/benchmark_questions_v1.json").is_file()
                  and (p / "notebooks/build04/NB09_v3.ipynb").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Set REGULATORY_AGENT_ROOT to the Regulatory_agent checkout.")
_repo_os.chdir(REPO_ROOT)

# Set RUN_PAID=True only after the free/offline preview below.
RUN_PAID = False
SELECT_QIDS = ["Q5", "Q9", "Q10"]
MODEL = "gpt-6-luna"  # v3's model; preserves model comparison
REASONING_EFFORT = "medium"
MAX_LLM_CALLS = 2
MAX_OUTPUT_TOKENS = 3200
SNIPPET_CHARS = 1700
DATA_DIR = __import__("pathlib").Path("data")
OUT_DIR = __import__("pathlib").Path("results/build04/v4_lean")

import json, re, os, time, hashlib, datetime
from pathlib import Path
from getpass import getpass

assert DATA_DIR.is_dir(), "Run from Regulatory_agent repository root."
chunk_path=__import__("pathlib").Path("data/processed/chunks_legal_aware.json")
benchmark_path=__import__("pathlib").Path("data/benchmark/benchmark_questions_v1.json")
assert chunk_path.is_file() and benchmark_path.is_file()
chunks=json.loads(chunk_path.read_text(encoding="utf-8"))
benchmark=json.loads(benchmark_path.read_text(encoding="utf-8"))
questions=benchmark["questions"]
assert len(chunks)==1383, "Frozen corpus size changed: investigate before running"
assert set(SELECT_QIDS)<=set(questions)
print("Corpus",len(chunks),"| questions",SELECT_QIDS,"| paid",RUN_PAID)


Corpus 1383 | questions ['Q5', 'Q9', 'Q10'] | paid True


## 1. Source routing — deterministic and free

The routes are *Article/Annex search targets*, not answer keys or score rubrics. Frozen candidate chunks are selected by article/location and indexed row labels, not by an LLM. If a provision cannot be found, it is recorded as missing and never filled from model memory. This is specifically a **targeted ablation** on the observed Q5/Q9/Q10 failures.


In [6]:
LAW_FILES = {
    "2023/915":"EU_2023_915_consolidated_2026-08-19.html",
    "2019/1793":"EU_2019_1793_consolidated_2026-06-30.html",
    "2017/625":"EU_2017_625_consolidated_2025-01-05.html",
    "178/2002":"EU_2002_178_consolidated_2026-01-01.html",
    "2019/1013":"EU_2019__1013_original.html",
}
ROUTES={
    "Q5":[("2019/1793","Annex II row 14"),("2019/1793","Article 3"),
          ("2019/1793","Article 4"),("2019/1793","Article 8"),
          ("2019/1793","Article 9"),("2019/1793","Article 10"),
          ("2019/1793","Article 11"),("2019/1793","Annex IV"),
          ("2017/625","Article 56"),("2019/1013","Article 1")],
    "Q9":[("2023/915","row 1.1.3"),("2023/915","Article 2"),
          ("2023/915","Article 3"),("2019/1793","Annex II row 14"),
          ("2019/1793","Article 9"),("2019/1793","Article 10"),
          ("2019/1793","Article 11"),("2019/1793","Annex IV"),
          ("2017/625","Article 56"),("2017/625","Article 66"),
          ("178/2002","Article 19"),("2019/1013","Article 1")],
    "Q10":[("2023/915","row 1.1.3"),("2023/915","Article 2"),
           ("2023/915","Article 3"),("2019/1793","Annex II row 14"),
           ("2019/1793","Article 9"),("2019/1793","Article 10"),
           ("2019/1793","Article 11"),("2019/1793","Annex IV"),
           ("2017/625","Article 56"),("2017/625","Article 66"),
           ("178/2002","Article 19")],
}

def matched_chunk(chunk,law,provision):
    if chunk.get("file") != LAW_FILES[law]: return False
    location=str(chunk.get("location") or "")
    article=str(chunk.get("article") or "")
    heading=str(chunk.get("heading") or "")
    text=str(chunk.get("text") or "")
    m=re.fullmatch(r"Article (\d+)",provision)
    if m:
        return article.lower()==provision.lower() or bool(re.search(r"(?:^|\b)Article\s+"+m.group(1)+r"(?!\d)",location,re.I))
    if provision=="row 1.1.3":
        # row label needs to be structural (not a prose mention).
        return ("1.1.3" in location and "annex" in location.lower()) or bool(re.search(r"Row:\s*1\.1\.3\b",text,re.I))
    if provision=="Annex II row 14":
        # Specific NB02 row-group chunk is preferable to incidental Annex text.
        return "annexII_row14_turkiye_figs" in str(chunk.get("chunk_id") or "") or (
            "row 14" in location.lower() and "annex ii" in location.lower())
    if provision=="Annex IV":
        h=(location+" "+heading).lower()
        return bool(re.search(r"\bannex iv\b",h)) and ("certificate" in (text+heading).lower() or "model" in h)
    return False

def retrieve_one(law,provision):
    matches=[ch for ch in chunks if matched_chunk(ch,law,provision)]
    # Reject ambiguous article-location chunks with no useful operative text.
    matches.sort(key=lambda ch:(0 if "logical row" in str(ch.get("location","")).lower() else 1,
                                -len(str(ch.get("text") or ""))))
    return matches[:1]

def to_evidence(ch,law,provision):
    chunk_id=str(ch.get("chunk_id") or "")
    return {"evidence_id":"E"+hashlib.sha256(chunk_id.encode()).hexdigest()[:10],
            "regulation_id":law,"requested_provision":provision,
            "source_file":ch.get("file"),"article":ch.get("article"),
            "location":ch.get("location"),"text":ch.get("text"),
            "chunk_id":chunk_id}

def collect_sources(qid):
    found={};missing=[]
    for law,provision in ROUTES[qid]:
        matches=retrieve_one(law,provision)
        if not matches:
            missing.append({"law":law,"provision":provision})
        for ch in matches:
            e=to_evidence(ch,law,provision)
            found.setdefault(e["evidence_id"],e)
    return list(found.values()),missing

def compact_sources(evidence):
    out=[]
    for e in evidence:
        t=str(e["text"] or "")
        shortened=(t if len(t)<=SNIPPET_CHARS else t[:1100]+" [...] "+t[-(SNIPPET_CHARS-1106):])
        out.append({"id":e["evidence_id"],"law":e["regulation_id"],
                    "article":e["article"],"location":e["location"],
                    "text":shortened})
    return out

for qid in SELECT_QIDS:
    ev,miss=collect_sources(qid)
    print(qid,"| unique chunks",len(ev),"| missing",miss)


Q5 | unique chunks 9 | missing [{'law': '2019/1793', 'provision': 'Annex IV'}]
Q9 | unique chunks 11 | missing [{'law': '2019/1793', 'provision': 'Annex IV'}]
Q10 | unique chunks 10 | missing [{'law': '2019/1793', 'provision': 'Annex IV'}]


## 2. Model: one answer + conditional repair

Only sourced claims should have evidence citations `[E...]`. A deterministic post-check detects fabricated citations and missing output sections. **A valid citation does not necessarily mean the statement is legally entailed by the cited text**; human Grounded/Correct evaluation remains required.


In [7]:
PROMPT = """You are an EU regulatory research assistant working ONLY from the supplied dated official-law extracts.
Answer the user's question directly and concisely. When the text supports a requirement, state it clearly;
do NOT turn an evidenced snapshot rule into an unknown simply because live applicability was not audited.
Never claim the snapshots prove current legal applicability. Do not invent a duty, number, article, date, or tariff classification.
Use exact [EvidenceID] citations for important legal claims, e.g. [E0123456789], from the given sources ONLY.
If a requested legal obligation lacks retrieved evidence, identify the missing provision instead of guessing.
Distinguish law from unknown consignment facts; detection alone does not prove exceedance.
Separate border-stage regulatory measures from post-market withdrawal/recall.
Answer with these four headings, exactly:
## Short answer
## Evidence-supported requirements
## Missing facts / legal verification
## Operator next steps
For legal-basis questions, map conclusions to exact Articles/Annexes; for procedures, group actions by duty.
Avoid repetitive blanket disclaimers. Do not use unpublished benchmark answers or evaluation rubrics.
"""

# Targeted development-set coverage *checks*, not training answers.
# Expected sections are provision categories, not prescribed substantive conclusions.
CRITICAL_PROVISIONS = {
    "Q5": [("2019/1793","Article 9"), ("2019/1793","Article 10"),
           ("2019/1793","Article 11"), ("2017/625","Article 56")],
    "Q9": [("2023/915","row 1.1.3"), ("2019/1793","Article 10"),
           ("2017/625","Article 66"), ("178/2002","Article 19")],
    "Q10": [("2023/915","row 1.1.3"), ("2019/1793","Article 10"),
            ("2019/1793","Article 11"), ("2017/625","Article 66")]
}

def structural_audit(answer,evidence,qid):
    allowed={e["evidence_id"] for e in evidence}
    cited=set(re.findall(r"\[(E[0-9a-f]{10})\]",answer))
    unknown=sorted(cited-allowed)
    sections=["## Short answer","## Evidence-supported requirements",
              "## Missing facts / legal verification","## Operator next steps"]
    missing_sections=[x for x in sections if x not in answer]
    # A demanded provision with actual retrieved evidence must appear in the answer's
    # citations; if not retrieved, report as legal evidence missing, not a model fault.
    uncited=[]
    for law,provision in CRITICAL_PROVISIONS.get(qid,[]):
        expected={e["evidence_id"] for e in evidence if e["regulation_id"]==law
                  and e["requested_provision"]==provision}
        if expected and not (expected & cited):uncited.append(f"{law} {provision}")
    return {"missing_sections":missing_sections,
            "invalid_citations":unknown,"has_citations":bool(cited),
            "uncited_retrieved_core_provisions":uncited,
            "cited_ids":sorted(cited),
            "passed":not unknown and bool(cited) and not missing_sections and not uncited}

def call_model(instruction,payload,client):
    response=client.responses.create(model=MODEL,
        reasoning={"effort":REASONING_EFFORT},
        instructions=instruction,input=json.dumps(payload,ensure_ascii=False),
        max_output_tokens=MAX_OUTPUT_TOKENS,store=False)
    usage=getattr(response,"usage",None)
    answer=(getattr(response,"output_text",None) or "").strip()
    if not answer:raise RuntimeError("Empty output: adjust max_output_tokens")
    return answer,{"input_tokens":int(getattr(usage,"input_tokens",0) or 0),
                   "output_tokens":int(getattr(usage,"output_tokens",0) or 0)}

def generate_answer(qid,evidence,missing,client):
    payload={"question":questions[qid],"case":"Türkiye-origin dried figs; B1 detected, concentration and stage unknown",
             "sources":compact_sources(evidence),"unretrieved_provisions":missing}
    first,u1=call_model(PROMPT,payload,client)
    audit=structural_audit(first,evidence,qid)
    used=[u1]; repaired=False
    if not audit["passed"] and len(used)<MAX_LLM_CALLS:
        repaired=True
        revised,u2=call_model(PROMPT+"\nRepair ONLY missing citations, relevant core source-coverage, and headings; do not invent sources.",
                              {**payload,"previous_answer":first,"audit":audit},client)
        used.append(u2)
        if structural_audit(revised,evidence,qid)["passed"]:
            first=revised
            audit=structural_audit(first,evidence,qid)
    return first,audit,used,repaired


## 3. Paid pilot (OFF by default) and immutable per-question outputs

To run paid calls, set `RUN_PAID=True` in the first code cell and rerun notebook top-to-bottom. The output folder is new and files are written with exclusive `x` mode so results will **never overwrite** v3. Do not add previous answers or grading rubric into the model prompt.


In [8]:
SETTINGS={"experiment":"NB09_v4_lean_q5q9q10", "benchmark_sha256":hashlib.sha256(benchmark_path.read_bytes()).hexdigest(),
          "chunks_sha256":hashlib.sha256(chunk_path.read_bytes()).hexdigest(),
          "model":MODEL,"reasoning_effort":REASONING_EFFORT,
          "max_llm_calls":MAX_LLM_CALLS,"source_policy":"five_law_snapshot_only",
          "route_status":"Q5_Q9_Q10_benchmark_exposed"}
OUT_DIR.mkdir(exist_ok=True,parents=True)
manifest=OUT_DIR/"manifest.json"
if RUN_PAID:
    if manifest.exists():
        assert json.loads(manifest.read_text(encoding="utf-8"))==SETTINGS, "Manifest changed: new experiment directory required"
    else:
        with manifest.open("x",encoding="utf-8") as f:json.dump(SETTINGS,f,indent=2)
    from openai import OpenAI
    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"]=getpass("OpenAI API key (hidden): ").strip()
    client=OpenAI()
    for qid in SELECT_QIDS:
        output=OUT_DIR/f"{qid}.json"
        if output.exists():
            print("SKIP existing",qid);continue
        evidence,missing=collect_sources(qid)
        assert evidence, "No retrieved sources; do not make a paid LLM call"
        started=time.perf_counter()
        answer,audit,usage,repaired=generate_answer(qid,evidence,missing,client)
        record={"qid":qid,"question":questions[qid],"experiment":SETTINGS,
                "final_output":answer,"evidence":evidence,"missing_provisions":missing,
                "llm_calls":len(usage),"search_calls":0,"local_provision_lookups":len(ROUTES[qid]),
                "usage":{"input_tokens":sum(u["input_tokens"] for u in usage),
                         "output_tokens":sum(u["output_tokens"] for u in usage)},
                "latency_seconds":round(time.perf_counter()-started,2),
                "repair_attempted":repaired,"audit":audit,
                "score_status":"human_review_pending",
                "created_utc":datetime.datetime.now(datetime.timezone.utc).isoformat()}
        with output.open("x",encoding="utf-8") as f:json.dump(record,f,ensure_ascii=False,indent=2)
        print(qid,"SAVED | LLM",record["llm_calls"],"| tokens",record["usage"],
              "| seconds",record["latency_seconds"],"| structural audit",audit["passed"])
else:
    print("DRY RUN: no API calls; inspect missing provisions before enabling RUN_PAID")


Q5 SAVED | LLM 2 | tokens {'input_tokens': 7553, 'output_tokens': 2559} | seconds 58.26 | structural audit True
Q9 SAVED | LLM 1 | tokens {'input_tokens': 4308, 'output_tokens': 1520} | seconds 19.43 | structural audit True
Q10 SAVED | LLM 1 | tokens {'input_tokens': 4061, 'output_tokens': 1486} | seconds 17.43 | structural audit True


## 4. Direct v3 ↔ v4 time/token comparison

A human should separately evaluate Correct / Grounded / Complete from the unmodified Notion reference. A structural-audit pass is **not** legal correctness and missing snippet evidence must not be invented.


In [9]:
try:
    import pandas as pd
except ImportError:
    pd=None
rows=[]
for qid in SELECT_QIDS:
    def read(path):return json.loads(path.read_text(encoding="utf-8")) if path.is_file() else {}
    v3=read(__import__("pathlib").Path("results/build04")/f"{qid}.json")
    v4=read(OUT_DIR/f"{qid}.json")
    def toks(x):
        u=x.get("usage") or {}
        return int(u.get("input_tokens") or 0)+int(u.get("output_tokens") or 0)
    rows.append({"qid":qid,"v3_calls":v3.get("llm_calls"),"v4_calls":v4.get("llm_calls"),
                 "v3_tokens":toks(v3) if v3 else None,"v4_tokens":toks(v4) if v4 else None,
                 "v3_seconds":v3.get("latency_seconds"),"v4_seconds":v4.get("latency_seconds"),
                 "v3_search_calls":v3.get("search_calls"),"v4_search_calls":v4.get("search_calls"),
                 "v4_local_lookups":v4.get("local_provision_lookups")})
if pd:display(pd.DataFrame(rows))
else:print(rows)


c:\Users\SAMSUNG\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


,qid,v3_calls,v4_calls,v3_tokens,v4_tokens,v3_seconds,v4_seconds,v3_search_calls,v4_search_calls,v4_local_lookups
0,Q5,8,2,72637,10112,260.34,58.26,8,0,10
1,Q9,7,1,70334,5828,259.15,19.43,8,0,12
2,Q10,7,1,58012,5547,229.37,17.43,8,0,11
